In [ ]:
import os
import json

import lusid
import lusidjam
from lusid.extensions import (
  SyncApiClientFactory,
  ArgsConfigurationLoader, 
  EnvironmentVariablesConfigurationLoader,
  SecretsFileConfigurationLoader
)
import lusid.api as lusid_api
from lusid.models import CreateRelationshipDefinitionRequest, CreateRelationshipRequest, DeleteRelationshipRequest

from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.cocoon import load_from_data_frame
from finbourne_sdk_utils.cocoon.cocoon_printer import (
    format_instruments_response,
    format_portfolios_response,
    format_transactions_response,
    format_quotes_response,
)

import pandas as pd


secrets_path = os.getenv("FBN_SECRETS_PATH")
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")


config_loaders=[
    ArgsConfigurationLoader(access_token=lusidjam.RefreshingToken(), app_name="LusidJupyterNotebook"),
    EnvironmentVariablesConfigurationLoader(),
    SecretsFileConfigurationLoader(api_secrets_file=secrets_path)
]

api_factory = SyncApiClientFactory(config_loaders=config_loaders)

print(api_factory.build(lusid_api.ApplicationMetadataApi).get_lusid_versions().build_version)


In [ ]:
# api definitions for the APIs we will be using
relationship_api = api_factory.build(lusid_api.RelationshipsApi)
relationship_definition_api = api_factory.build(lusid_api.RelationshipDefinitionsApi)
portfolios_api = api_factory.build(lusid_api.PortfoliosApi)

In [ ]:
scope = "relationships-example"
relationship_code = "master"
subaccount_relationship_code = "subaccount1"

# Portfolios


* `master` fund with a relation to 3 sub-funds
* `subaccount1` related to `subaccount1-monthly-postions` representing a version containing monthly positions 
* `subaccount1` related to `subaccount1-abor` representing an ABOR view


In [ ]:
portfolios_df = pd.DataFrame(
    data=[
        ["master", "master"],
        ["subaccount1", "subaccount1"],
        ["subaccount2", "subaccount2"],
        ["subaccount3", "subaccount3"],
        ["subaccount1-monthly-postions", "subaccount1-monthly-postions"],
        ["subaccount1-abor", "subaccount1-abor"]
    ], columns=["portfolio_code", "portfolio_name"]
)

result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=portfolios_df,
    mapping_required={
        "code": "portfolio_code",
        "display_name": "portfolio_name",
        "base_currency": "$GBP",
    },
    mapping_optional={
        "created": "$2020-01-01T00:00:00+00:00"
    },
    file_type="portfolios",
)

succ, failed = format_portfolios_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed)}])

# Create Relationship definition

In [ ]:
def create_relationship_definition(code: str, display_name: str, outward_description: str, inward_description: str):
    
    try:
        result = relationship_definition_api.create_relationship_definition(
            create_relationship_definition_request=CreateRelationshipDefinitionRequest(
            scope=scope,
            code=code,
            sourceEntityType="Portfolio",
            targetEntityType="Portfolio",
            displayName=display_name,
            outwardDescription=outward_description,
            inwardDescription=inward_description,
            lifeTime="TimeVariant",
            relationshipCardinality=None) 
        )
        return result
    except lusid.ApiException as e:
        body = json.loads(e.body)
        if body["code"] != 667:  # RelationDefinitionAlreadyExists
            print(body)
        else:
            print(f"relation {scope}/{code} already exists")


In [ ]:
create_relationship_definition("subfund", "Master fund link to sub-fund", "parent of", "sub-fund of")

In [ ]:
create_relationship_definition("monthly-positions", "Link to fund containing monthly positions", "daily transactions of", "has monthly positions of")

In [ ]:
create_relationship_definition("abor", "Link to fund containing ABOR", "IBOR of", "ABOR of")

In [ ]:

def create_portfolio_relationship(relation_code: str, from_portfolio: str, to_portfolio: str):
    res = relationship_api.create_relationship(
        scope=scope,
        code=relation_code,
        create_relationship_request=CreateRelationshipRequest(
        sourceEntityId={
            "scope": scope,
            "code": from_portfolio
        },
        targetEntityId={
            "scope": scope,
            "code": to_portfolio
        },
        effectiveFrom="2020-01-01T00:00:00+00:00",  # Must match or be after portfolio creation date
        effectiveUntil=None
    ))
    return res

def delete_portfolio_relationship(relationship_code: str, from_portfolio: str, to_portfolio: str):
    res = relationship_api.delete_relationship(scope=scope,code=relationship_code, delete_relationship_request=DeleteRelationshipRequest(
        sourceEntityId={
            "scope": scope,
            "code": from_portfolio
        },
        targetEntityId={
            "scope": scope,
            "code": to_portfolio
        },
        effectiveFrom="2020-01-01T00:00:00+00:00",
        effectiveUntil=None
    ))


In [ ]:
create_portfolio_relationship("subfund", relationship_code, "subaccount1")

In [ ]:
create_portfolio_relationship("subfund", relationship_code, "subaccount2")

In [ ]:
create_portfolio_relationship("subfund", relationship_code, "subaccount3")

In [ ]:
create_portfolio_relationship("monthly-positions", "subaccount1", "subaccount1-monthly-postions")

In [ ]:
create_portfolio_relationship("abor", "subaccount1", "subaccount1-abor")

In [ ]:
sub_funds = portfolios_api.get_portfolio_relationships(scope=scope, code=relationship_code, effective_at="2020-01-01T00:00:00+00:00")
lusid_response_to_data_frame(sub_funds.values)

## `subaccount1` relationships

Get the relations for `subaccount1`. There is the Relation `in` from portfolio `master`, and 2 `out` Relations to the `subaccount1-monthly-postions` and `subaccount1-abor` portfolios

In [ ]:
sub_account1_relationships = portfolios_api.get_portfolio_relationships(scope=scope, code="subaccount1", effective_at="2020-01-01T00:00:00+00:00")
display(lusid_response_to_data_frame(sub_account1_relationships.values))

## Get the related monthly positions for `subaccount1`

In [ ]:
monthly_relation = list(filter(lambda relation: relation.related_entity.entity_id["code"] == "subaccount1-monthly-postions", sub_account1_relationships.values))

portfolio = portfolios_api.get_portfolio(
    scope=monthly_relation[0].related_entity.entity_id["scope"],
    code=monthly_relation[0].related_entity.entity_id["code"],
)

pd.DataFrame([{
    "scope": portfolio.id.scope,
    "code": portfolio.id.code,
    "display_name": portfolio.display_name
}])


In [ ]:
def relationship_filter(relationship):
    return relationship.related_entity.entity_id["code"] == relationship_code and relationship.traversal_direction == "In"

group_relation = list(filter(relationship_filter, sub_account1_relationships.values))

portfolio = portfolios_api.get_portfolio(
    scope=group_relation[0].related_entity.entity_id["scope"],
    code=group_relation[0].related_entity.entity_id["code"],
)

pd.DataFrame([{
    "scope": portfolio.id.scope,
    "code": portfolio.id.code,
    "display_name": portfolio.display_name
}])

In [ ]:
  
delete_portfolio_relationship("subfund", relationship_code, "subaccount1")
delete_portfolio_relationship("subfund", relationship_code, "subaccount2")
delete_portfolio_relationship("subfund", relationship_code, "subaccount3")
delete_portfolio_relationship("monthly-positions", "subaccount1", "subaccount1-monthly-postions")
delete_portfolio_relationship("abor", "subaccount1", "subaccount1-abor")

relationship_defs_to_delete = [
  "subfund",
  "monthly-positions",
  "abor"
]

for relationship_def_code in relationship_defs_to_delete:
  deleted_def = relationship_definition_api.delete_relationship_definition(scope=scope,code=relationship_def_code)
